In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Install All Dependencies
# Runtime: T4 GPU | ~3-4 minutes
# ═══════════════════════════════════════════════════════════════

!pip install -q colpali-engine chromadb sentence-transformers \
               pdf2image pymupdf accelerate colorlog \
               arxiv bitsandbytes gradio

!apt-get install -y poppler-utils 2>/dev/null

# ─── Verify everything ───
import torch
import numpy as np
import transformers
import peft

print("=" * 55)
print("  PACKAGE VERIFICATION")
print("=" * 55)
print(f"  numpy              : {np.__version__}")
print(f"  torch              : {torch.__version__}")
print(f"  transformers       : {transformers.__version__}")
print(f"  peft               : {peft.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  GPU                : {gpu_name}")
    print(f"  VRAM               : {vram:.1f} GB")
else:
    print("  ⚠️  No GPU! Go to Runtime → Change type → T4")

try:
    from colpali_engine.models import ColPali, ColPaliProcessor
    print("  colpali-engine     : ✅")
except Exception as e:
    print(f"  colpali-engine     : ❌ {e}")

try:
    from sentence_transformers import SentenceTransformer
    print("  sentence-transformers : ✅")
except Exception as e:
    print(f"  sentence-transformers : ❌ {e}")

try:
    import chromadb
    print(f"  chromadb           : {chromadb.__version__} ✅")
except Exception as e:
    print(f"  chromadb           : ❌ {e}")

try:
    import fitz
    print(f"  PyMuPDF            : {fitz.__version__} ✅")
except Exception as e:
    print(f"  PyMuPDF            : ❌ {e}")

try:
    from pdf2image import convert_from_path
    print("  pdf2image          : ✅")
except Exception as e:
    print(f"  pdf2image          : ❌ {e}")

import gc
import json
import os
import time
from pathlib import Path
from PIL import Image

print("\n✅ CELL 1 COMPLETE — all packages ready!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
  PACKAGE VERIFICATION
  numpy              : 2.0.2
  torch              : 2.10.0+cu128
  transformers       : 5.9.0
  peft               : 0.19.1
  GPU                : Tesla T4
  VRAM               : 14.6 GB
  colpali-engine     : ✅
  sentence-transformers : ✅
  chromadb           : 1.5.9 ✅
  PyMuPDF            : 1.27.2.3 ✅
  pdf2image          : ✅

✅ CELL 1 COMPLETE — all packages ready!


In [3]:

# ═══════════════════════════════════════════════════════════════
# CELL 2: Create Project Directories + Config
# ═══════════════════════════════════════════════════════════════

import os, json

# ─── Directories ───
DIRS = {
    "raw":      "data/raw",
    "pages":    "data/parsed/pages",
    "markdown": "data/parsed/markdown",
    "npy":      "data/indices/multivectors",
    "chroma":   "data/indices/chroma_index",
    "indices":  "data/indices",
    "output":   "outputs",
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  📁 {name:15s} → {path}")

# ─── Paper list (10 arXiv Vision Transformer papers) ───
PAPERS = {
    "2010.11929": "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale",
    "2106.10270": "Training data-efficient image transformers & distillation through attention",
    "2205.01580": "A Battle of Architectures: ViT, ResNet and Hybrid in 3D Medical Image Segmentation",
    "2108.00102": "DeepViT: Towards Deeper Vision Transformer",
    "2203.14465": "EfficientFormer: Vision Transformers at MobileNet Speed",
    "2211.11505": "Scaling Vision Transformers to 22 Billion Parameters",
    "2303.15326": "A Survey on Vision Transformer",
    "2106.14881": "Swin Transformer V2: Scaling Up Capacity and Resolution",
    "2111.09883": "Aggregating Nested Transformers",
    "2204.00636": "Egocentric Video Understanding Using Vision Transformers",
}

# ─── Save config ───
config = {
    "papers": PAPERS,
    "dirs": DIRS,
    "models": {
        "colpali": "vidore/colpali-v1.2",
        "scincl":  "malteos/scincl",
        "qwen2vl": "Qwen/Qwen2-VL-2B-Instruct",
    },
    "retrieval": {
        "top_k":           5,
        "colpali_weight":  0.7,
        "scincl_weight":   0.3,
    },
    "parsing": {"dpi": 200},
}

with open("data/config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\n  📄 Config saved → data/config.json")
print(f"\n✅ CELL 2 COMPLETE — {len(PAPERS)} papers configured!")

  📁 raw             → data/raw
  📁 pages           → data/parsed/pages
  📁 markdown        → data/parsed/markdown
  📁 npy             → data/indices/multivectors
  📁 chroma          → data/indices/chroma_index
  📁 indices         → data/indices
  📁 output          → outputs

  📄 Config saved → data/config.json

✅ CELL 2 COMPLETE — 10 papers configured!


In [4]:

# ═══════════════════════════════════════════════════════════════
# CELL 3: Download 10 arXiv PDFs
# Time: ~3-5 min (depends on internet)
# ═══════════════════════════════════════════════════════════════

import requests
import time
import os
import json

PAPERS = {
    "2010.11929": "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale",
    "2106.10270": "Training data-efficient image transformers & distillation through attention",
    "2205.01580": "A Battle of Architectures: ViT, ResNet and Hybrid in 3D Medical Image Segmentation",
    "2108.00102": "DeepViT: Towards Deeper Vision Transformer",
    "2203.14465": "EfficientFormer: Vision Transformers at MobileNet Speed",
    "2211.11505": "Scaling Vision Transformers to 22 Billion Parameters",
    "2303.15326": "A Survey on Vision Transformer",
    "2106.14881": "Swin Transformer V2: Scaling Up Capacity and Resolution",
    "2111.09883": "Aggregating Nested Transformers",
    "2204.00636": "Egocentric Video Understanding Using Vision Transformers",
}

RAW_DIR = "data/raw"

print("=" * 60)
print(f"  DOWNLOADING {len(PAPERS)} arXiv PDFs")
print("=" * 60)

download_results = []
success_count = 0
fail_count = 0

for i, (arxiv_id, title) in enumerate(PAPERS.items()):
    pdf_path = os.path.join(RAW_DIR, f"{arxiv_id}.pdf")

    # Skip if already downloaded
    if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 10000:
        size_mb = os.path.getsize(pdf_path) / 1e6
        print(f"  [{i+1:2d}/{len(PAPERS)}] SKIP (exists): {arxiv_id} ({size_mb:.1f} MB)")
        download_results.append({
            "arxiv_id": arxiv_id, "title": title,
            "status": "exists", "pdf_path": pdf_path
        })
        success_count += 1
        continue

    url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"
    print(f"  [{i+1:2d}/{len(PAPERS)}] Downloading: {arxiv_id}")
    print(f"           {title[:60]}...")

    try:
        r = requests.get(url, timeout=120, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200 and len(r.content) > 10000:
            with open(pdf_path, "wb") as f:
                f.write(r.content)
            size_mb = len(r.content) / 1e6
            print(f"           ✅ {size_mb:.1f} MB")
            download_results.append({
                "arxiv_id": arxiv_id, "title": title,
                "status": "success", "pdf_path": pdf_path
            })
            success_count += 1
        else:
            print(f"           ❌ HTTP {r.status_code} or tiny file")
            download_results.append({
                "arxiv_id": arxiv_id, "title": title,
                "status": "failed", "error": f"HTTP {r.status_code}"
            })
            fail_count += 1
    except Exception as e:
        print(f"           ❌ {e}")
        download_results.append({
            "arxiv_id": arxiv_id, "title": title,
            "status": "failed", "error": str(e)
        })
        fail_count += 1

    # Rate limit — arXiv needs 3 sec gap
    time.sleep(3)

print(f"\n{'='*60}")
print(f"  DOWNLOAD COMPLETE: {success_count} success, {fail_count} failed")
print(f"{'='*60}")

# Save download results
with open("data/download_results.json", "w") as f:
    json.dump(download_results, f, indent=2)

print("✅ CELL 3 COMPLETE!")

  DOWNLOADING 10 arXiv PDFs
  [ 1/10] Downloading: 2010.11929
           An Image is Worth 16x16 Words: Transformers for Image Recogn...
           ✅ 3.7 MB
  [ 2/10] Downloading: 2106.10270
           Training data-efficient image transformers & distillation th...
           ✅ 0.9 MB
  [ 3/10] Downloading: 2205.01580
           A Battle of Architectures: ViT, ResNet and Hybrid in 3D Medi...
           ✅ 0.2 MB
  [ 4/10] Downloading: 2108.00102
           DeepViT: Towards Deeper Vision Transformer...
           ✅ 1.3 MB
  [ 5/10] Downloading: 2203.14465
           EfficientFormer: Vision Transformers at MobileNet Speed...
           ✅ 0.8 MB
  [ 6/10] Downloading: 2211.11505
           Scaling Vision Transformers to 22 Billion Parameters...
           ✅ 6.3 MB
  [ 7/10] Downloading: 2303.15326
           A Survey on Vision Transformer...
           ✅ 2.0 MB
  [ 8/10] Downloading: 2106.14881
           Swin Transformer V2: Scaling Up Capacity and Resolution...
           ✅ 2.7 MB
  [ 9/

In [5]:

# ═══════════════════════════════════════════════════════════════
# CELL 4: Parse PDFs — Extract Text + Page Images
# Time: ~3-5 minutes
# ═══════════════════════════════════════════════════════════════

import os
import json
import time
import fitz  # PyMuPDF
from pdf2image import convert_from_path
from pathlib import Path

# ─── Config ───
PAPERS = {
    "2010.11929": "An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale",
    "2106.10270": "Training data-efficient image transformers & distillation through attention",
    "2205.01580": "A Battle of Architectures: ViT, ResNet and Hybrid in 3D Medical Image Segmentation",
    "2108.00102": "DeepViT: Towards Deeper Vision Transformer",
    "2203.14465": "EfficientFormer: Vision Transformers at MobileNet Speed",
    "2211.11505": "Scaling Vision Transformers to 22 Billion Parameters",
    "2303.15326": "A Survey on Vision Transformer",
    "2106.14881": "Swin Transformer V2: Scaling Up Capacity and Resolution",
    "2111.09883": "Aggregating Nested Transformers",
    "2204.00636": "Egocentric Video Understanding Using Vision Transformers",
}

RAW_DIR      = "data/raw"
PAGES_DIR    = "data/parsed/pages"
MARKDOWN_DIR = "data/parsed/markdown"
DPI          = 200

# ─── Load download results ───
with open("data/download_results.json", "r") as f:
    download_results = json.load(f)

# Only parse successfully downloaded papers
successful = [d for d in download_results if d["status"] in ["success", "exists"]]

print("=" * 60)
print(f"  CELL 4: PARSING {len(successful)} PDFs")
print(f"  Mode: Text (PyMuPDF) + Images (pdf2image @ {DPI} DPI)")
print("=" * 60)

doc_mapping   = {}
page_metadata = {}
total_pages   = 0
parse_start   = time.time()

for idx, dl in enumerate(successful):
    arxiv_id = dl["arxiv_id"]
    title    = PAPERS.get(arxiv_id, arxiv_id)
    pdf_path = dl["pdf_path"]

    print(f"\n  [{idx+1:2d}/{len(successful)}] Parsing: {arxiv_id}")
    print(f"            {title[:55]}...")

    # ─── Text Extraction (PyMuPDF) ───
    try:
        doc = fitz.open(pdf_path)
        page_texts = []
        for page in doc:
            text = page.get_text("text")
            page_texts.append(text)
        num_pages = len(page_texts)
        doc.close()
        print(f"            📄 Text: {num_pages} pages extracted")
    except Exception as e:
        print(f"            ❌ Text extraction failed: {e}")
        page_texts = []
        num_pages  = 0

    # ─── Save Markdown ───
    md_path = os.path.join(MARKDOWN_DIR, f"{arxiv_id}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write(f"arXiv ID: {arxiv_id}\n\n")
        for i, text in enumerate(page_texts):
            f.write(f"## Page {i+1}\n\n{text}\n\n---\n\n")

    # ─── Image Extraction (pdf2image) ───
    page_images = []
    try:
        images = convert_from_path(pdf_path, dpi=DPI)
        for i, img in enumerate(images):
            img_path = os.path.join(PAGES_DIR, f"{arxiv_id}_page_{i+1}.png")
            img.save(img_path, "PNG")
            page_images.append(img_path)
        print(f"            🖼️  Images: {len(page_images)} pages saved")
    except Exception as e:
        print(f"            ⚠️  Image extraction error: {e}")
        print(f"            Text only mode!")

    # ─── Build Metadata ───
    doc_mapping[arxiv_id] = {
        "arxiv_id":      arxiv_id,
        "title":         title,
        "num_pages":     num_pages,
        "page_images":   page_images,
        "markdown_path": md_path,
        "status":        "success",
    }

    for i in range(num_pages):
        page_key = f"{arxiv_id}_page_{i+1}"
        page_metadata[page_key] = {
            "doc_id":      arxiv_id,
            "page_num":    i + 1,
            "image_path":  page_images[i] if i < len(page_images) else "",
            "text":        page_texts[i]  if i < len(page_texts)  else "",
            "paper_title": title,
        }

    total_pages += num_pages
    print(f"            ✅ Done! ({num_pages} pages)")

# ─── Save Metadata Files ───
with open("data/indices/doc_mapping.json", "w", encoding="utf-8") as f:
    json.dump(doc_mapping, f, indent=2, ensure_ascii=False)

with open("data/indices/page_metadata.json", "w", encoding="utf-8") as f:
    json.dump(page_metadata, f, indent=2, ensure_ascii=False)

parse_time = time.time() - parse_start

print(f"\n{'='*60}")
print(f"  PARSE COMPLETE")
print(f"  Papers parsed : {len(doc_mapping)}")
print(f"  Total pages   : {total_pages}")
print(f"  Time taken    : {parse_time:.1f}s")
print(f"{'='*60}")
print("✅ CELL 4 COMPLETE!")

  CELL 4: PARSING 10 PDFs
  Mode: Text (PyMuPDF) + Images (pdf2image @ 200 DPI)

  [ 1/10] Parsing: 2010.11929
            An Image is Worth 16x16 Words: Transformers for Image R...
            📄 Text: 22 pages extracted
            🖼️  Images: 22 pages saved
            ✅ Done! (22 pages)

  [ 2/10] Parsing: 2106.10270
            Training data-efficient image transformers & distillati...
            📄 Text: 16 pages extracted
            🖼️  Images: 16 pages saved
            ✅ Done! (16 pages)

  [ 3/10] Parsing: 2205.01580
            A Battle of Architectures: ViT, ResNet and Hybrid in 3D...
            📄 Text: 3 pages extracted
            🖼️  Images: 3 pages saved
            ✅ Done! (3 pages)

  [ 4/10] Parsing: 2108.00102
            DeepViT: Towards Deeper Vision Transformer...
            📄 Text: 37 pages extracted
            🖼️  Images: 37 pages saved
            ✅ Done! (37 pages)

  [ 5/10] Parsing: 2203.14465
            EfficientFormer: Vision Transformers at MobileNet

In [1]:

# ═══════════════════════════════════════════════════════════════
# FRESH START CELL: Reinstall + ColPali Embedding
# Run this RIGHT AFTER runtime restart!
# ═══════════════════════════════════════════════════════════════

# ─── 1. Quick installs ───
import subprocess
subprocess.run(["pip", "install", "-q", "colpali-engine",
                "pymupdf", "pdf2image", "chromadb",
                "sentence-transformers", "accelerate"], check=False)

import os, gc, json, time
import numpy as np
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor

# ─── 2. Set VRAM fragmentation fix ───
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ─── 3. Check clean VRAM ───
free_vram = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_allocated()) / 1024**3
print("=" * 60)
print("  FRESH START — ColPali Embedding")
print(f"  Device    : {torch.cuda.get_device_name(0)}")
print(f"  VRAM free : {free_vram:.1f} GB")
print("=" * 60)

if free_vram < 10.0:
    print("⚠️  WARNING: Less than 10GB free — runtime may not be clean!")
    print("   Please do Runtime → Restart Session again!")
else:
    print("✅ VRAM is clean — good to go!")

# ─── 4. Config ───
NPY_DIR   = "data/indices/multivectors"

# ─── 5. Load metadata ───
with open("data/indices/page_metadata.json", "r") as f:
    page_metadata = json.load(f)
print(f"\n  Total pages in metadata: {len(page_metadata)}")

# ─── 6. Load ColPali (float16, no quantization needed on clean VRAM) ───
print("\n  Loading ColPali model (vidore/colpali-v1.2)...")

colpali_model = ColPali.from_pretrained(
    "vidore/colpali-v1.2",
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,
)
colpali_processor = ColPaliProcessor.from_pretrained("vidore/colpali-v1.2")
colpali_model.eval()

vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"  ✅ ColPali loaded! VRAM used: {vram_used:.2f} GB")

# ─── 7. Check already embedded ───
existing_npy = set(
    f.replace(".npy", "")
    for f in os.listdir(NPY_DIR)
    if f.endswith(".npy") and not f.endswith(".meta.npy")
)
print(f"\n  Already embedded : {len(existing_npy)} pages")
print(f"  To embed         : {len(page_metadata) - len(existing_npy)} pages")

# ─── 8. Build list ───
all_keys   = []
all_images = []

for page_key, meta in page_metadata.items():
    if page_key in existing_npy:
        continue
    img_path = meta.get("image_path", "")
    if not img_path or not os.path.exists(img_path):
        continue
    all_keys.append(page_key)
    all_images.append(img_path)

print(f"  Valid images to embed: {len(all_images)}")
print(f"\n  Starting embedding... (~15-20 min)\n")

# ─── 9. Embed one by one ───
embedded_count = 0
error_count    = 0
emb_start      = time.time()

for i, (page_key, img_path) in enumerate(zip(all_keys, all_images)):
    try:
        img   = Image.open(img_path).convert("RGB")
        batch = colpali_processor.process_images(images=[img])
        batch = {k: v.to(colpali_model.device) for k, v in batch.items()}

        with torch.no_grad():
            embeddings = colpali_model(**batch)

        vectors  = embeddings[0].cpu().float().numpy()
        npy_path = os.path.join(NPY_DIR, f"{page_key}.npy")
        np.save(npy_path, vectors)
        embedded_count += 1

        del batch, embeddings, img
        torch.cuda.empty_cache()

        # Progress every 20 pages
        if (i + 1) % 20 == 0 or (i + 1) == len(all_images):
            elapsed = time.time() - emb_start
            rate    = embedded_count / max(elapsed, 1)
            eta     = (len(all_images) - (i+1)) / max(rate, 0.01) / 60
            pct     = (i + 1) / len(all_images) * 100
            vram    = torch.cuda.memory_allocated() / 1024**3
            print(f"  [{i+1:3d}/{len(all_images)}] {pct:.0f}% | "
                  f"embedded: {embedded_count} | "
                  f"rate: {rate:.2f} p/s | "
                  f"ETA: {eta:.1f}min | "
                  f"VRAM: {vram:.1f}GB")

    except torch.cuda.OutOfMemoryError:
        print(f"  ⚠️  OOM on {page_key} — skipping...")
        torch.cuda.empty_cache()
        gc.collect()
        error_count += 1

    except Exception as e:
        print(f"  ❌ {page_key}: {e}")
        error_count += 1
        torch.cuda.empty_cache()

# ─── 10. Unload ColPali ───
print("\n  Unloading ColPali...")
del colpali_model, colpali_processor
gc.collect()
torch.cuda.empty_cache()

emb_time  = time.time() - emb_start
npy_files = [f for f in os.listdir(NPY_DIR)
             if f.endswith(".npy") and not f.endswith(".meta.npy")]

print(f"\n{'='*60}")
print(f"  COLPALI EMBEDDING COMPLETE")
print(f"  Embedded  : {embedded_count} pages")
print(f"  Skipped   : {len(existing_npy)} already done")
print(f"  Errors    : {error_count}")
print(f"  Time      : {emb_time/60:.1f} minutes")
print(f"  .npy files: {len(npy_files)}")
print(f"{'='*60}")
print("✅ CELL 5 COMPLETE!")

  FRESH START — ColPali Embedding
  Device    : Tesla T4
  VRAM free : 14.6 GB
✅ VRAM is clean — good to go!

  Total pages in metadata: 182

  Loading ColPali model (vidore/colpali-v1.2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] ColPali LOAD REPORT from: vidore/colpali-v1.2
Key                                                                               | Status     | 
----------------------------------------------------------------------------------+------------+-
model.language_model.model.layers.{0...17}.mlp.gate_proj.lora_B.default.weight    | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.o_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.down_proj.lora_B.default.weight    | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.q_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.k_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.up_proj.lo

  ✅ ColPali loaded! VRAM used: 5.52 GB

  Already embedded : 0 pages
  To embed         : 182 pages
  Valid images to embed: 182

  Starting embedding... (~15-20 min)

  [ 20/182] 11% | embedded: 20 | rate: 1.75 p/s | ETA: 1.5min | VRAM: 5.5GB
  [ 40/182] 22% | embedded: 40 | rate: 1.88 p/s | ETA: 1.3min | VRAM: 5.5GB
  [ 60/182] 33% | embedded: 60 | rate: 1.92 p/s | ETA: 1.1min | VRAM: 5.5GB
  [ 80/182] 44% | embedded: 80 | rate: 1.92 p/s | ETA: 0.9min | VRAM: 5.5GB
  [100/182] 55% | embedded: 100 | rate: 1.92 p/s | ETA: 0.7min | VRAM: 5.5GB
  [120/182] 66% | embedded: 120 | rate: 1.82 p/s | ETA: 0.6min | VRAM: 5.5GB
  [140/182] 77% | embedded: 140 | rate: 1.85 p/s | ETA: 0.4min | VRAM: 5.5GB
  [160/182] 88% | embedded: 160 | rate: 1.87 p/s | ETA: 0.2min | VRAM: 5.5GB
  [180/182] 99% | embedded: 180 | rate: 1.88 p/s | ETA: 0.0min | VRAM: 5.5GB
  [182/182] 100% | embedded: 182 | rate: 1.88 p/s | ETA: 0.0min | VRAM: 5.5GB

  Unloading ColPali...

  COLPALI EMBEDDING COMPLETE
  Embedded 

In [2]:

# ═══════════════════════════════════════════════════════════════
# CELL 6: SciNCL Text Embedding → ChromaDB
# Time: ~3-5 minutes for 182 pages
# ═══════════════════════════════════════════════════════════════

import os
import gc
import json
import time
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

# ─── Config ───
CHROMA_DIR = "data/indices/chroma_index"

# ─── Load metadata ───
with open("data/indices/page_metadata.json", "r") as f:
    page_metadata = json.load(f)

print("=" * 60)
print("  CELL 6: SciNCL TEXT EMBEDDING → ChromaDB")
print(f"  Total pages : {len(page_metadata)}")
print(f"  Device      : {torch.cuda.get_device_name(0)}")
free_vram = (torch.cuda.get_device_properties(0).total_memory
             - torch.cuda.memory_allocated()) / 1024**3
print(f"  VRAM free   : {free_vram:.1f} GB")
print("=" * 60)

# ─── Load SciNCL model ───
print("\n  Loading SciNCL model (malteos/scincl)...")
scincl_model = SentenceTransformer("malteos/scincl", device="cuda")
vram_used = torch.cuda.memory_allocated() / 1024**3
print(f"  ✅ SciNCL loaded! VRAM used: {vram_used:.2f} GB")

# ─── Init ChromaDB ───
print("\n  Initializing ChromaDB...")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if exists (fresh start)
try:
    chroma_client.delete_collection("sci_rag_pages")
    print("  Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name="sci_rag_pages",
    metadata={"hnsw:space": "cosine"},
)
print(f"  ✅ ChromaDB collection created!")

# ─── Prepare texts ───
texts_to_embed = []
metas_to_embed = []
ids_to_embed   = []

for page_key, meta in page_metadata.items():
    text = meta.get("text", "")
    if not text or len(text.strip()) < 20:
        continue
    texts_to_embed.append(text[:512])
    metas_to_embed.append({
        "doc_id":      meta["doc_id"],
        "page_num":    str(meta["page_num"]),
        "paper_title": meta.get("paper_title", meta["doc_id"]),
    })
    ids_to_embed.append(page_key)

print(f"\n  Pages with valid text : {len(texts_to_embed)}")
print(f"  Pages skipped (empty) : {len(page_metadata) - len(texts_to_embed)}")
print(f"\n  Starting embedding...\n")

# ─── Batch encode + store ───
BATCH      = 32
count      = 0
scincl_start = time.time()

for i in range(0, len(texts_to_embed), BATCH):
    batch_texts = texts_to_embed[i:i+BATCH]
    batch_ids   = ids_to_embed[i:i+BATCH]
    batch_metas = metas_to_embed[i:i+BATCH]

    # Encode
    embeddings = scincl_model.encode(
        batch_texts,
        show_progress_bar=False,
        batch_size=BATCH,
        convert_to_numpy=True,
    )

    # Store in ChromaDB
    collection.upsert(
        ids=batch_ids,
        embeddings=embeddings.tolist(),
        documents=[t[:500] for t in batch_texts],
        metadatas=batch_metas,
    )

    count += len(batch_ids)
    elapsed = time.time() - scincl_start
    rate    = count / max(elapsed, 1)
    eta     = (len(texts_to_embed) - count) / max(rate, 0.1) / 60
    print(f"  [{count:3d}/{len(texts_to_embed)}] "
          f"rate: {rate:.1f} p/s | "
          f"ETA: {eta:.1f} min")

# ─── Unload SciNCL ───
print("\n  Unloading SciNCL...")
del scincl_model
gc.collect()
torch.cuda.empty_cache()

scincl_time = time.time() - scincl_start

print(f"\n{'='*60}")
print(f"  SciNCL EMBEDDING COMPLETE")
print(f"  Embedded      : {count} pages")
print(f"  ChromaDB total: {collection.count()} entries")
print(f"  Time          : {scincl_time/60:.1f} minutes")
print(f"{'='*60}")
print("✅ CELL 6 COMPLETE!")

  CELL 6: SciNCL TEXT EMBEDDING → ChromaDB
  Total pages : 182
  Device      : Tesla T4
  VRAM free   : 14.6 GB

  Loading SciNCL model (malteos/scincl)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

  ✅ SciNCL loaded! VRAM used: 0.42 GB

  Initializing ChromaDB...
  ✅ ChromaDB collection created!

  Pages with valid text : 182
  Pages skipped (empty) : 0

  Starting embedding...

  [ 32/182] rate: 32.0 p/s | ETA: 0.1 min
  [ 64/182] rate: 51.2 p/s | ETA: 0.0 min
  [ 96/182] rate: 56.7 p/s | ETA: 0.0 min
  [128/182] rate: 58.2 p/s | ETA: 0.0 min
  [160/182] rate: 55.0 p/s | ETA: 0.0 min
  [182/182] rate: 54.3 p/s | ETA: 0.0 min

  Unloading SciNCL...

  SciNCL EMBEDDING COMPLETE
  Embedded      : 182 pages
  ChromaDB total: 182 entries
  Time          : 0.1 minutes
✅ CELL 6 COMPLETE!


In [3]:

# ═══════════════════════════════════════════════════════════════
# CELL 7: Save All Indices + Create Download ZIPs
# Time: ~2 minutes
# ═══════════════════════════════════════════════════════════════

import os
import gc
import json
import time
import zipfile
import torch

# ─── Load metadata ───
with open("data/indices/page_metadata.json", "r") as f:
    page_metadata = json.load(f)

with open("data/indices/doc_mapping.json", "r") as f:
    doc_mapping = json.load(f)

print("=" * 60)
print("  CELL 7: SAVE + ZIP ALL INDICES")
print("=" * 60)

# ─── 1. Count .npy files ───
NPY_DIR   = "data/indices/multivectors"
npy_files = [f for f in os.listdir(NPY_DIR)
             if f.endswith(".npy") and not f.endswith(".meta.npy")]
print(f"\n  [1/4] Index Check:")
print(f"  ColPali .npy files : {len(npy_files)}")
print(f"  Page metadata      : {len(page_metadata)}")
print(f"  Doc mapping        : {len(doc_mapping)}")

# ─── 2. Save summary.json ───
print(f"\n  [2/4] Saving summary...")

total_pages = sum(d["num_pages"] for d in doc_mapping.values())
npy_size_mb = sum(
    os.path.getsize(os.path.join(NPY_DIR, f)) / 1024**2
    for f in npy_files
)

summary = {
    "created_at"     : time.strftime("%Y-%m-%d %H:%M:%S"),
    "num_papers"     : len(doc_mapping),
    "total_pages"    : total_pages,
    "colpali_files"  : len(npy_files),
    "colpali_size_mb": round(npy_size_mb, 2),
    "chroma_entries" : len(page_metadata),
    "models_used"    : {
        "colpali" : "vidore/colpali-v1.2",
        "scincl"  : "malteos/scincl",
    },
    "papers": {
        arxiv_id: {
            "title"    : info["title"],
            "num_pages": info["num_pages"],
        }
        for arxiv_id, info in doc_mapping.items()
    }
}

with open("data/indices/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"  ✅ Summary saved!")
print(f"     Papers     : {summary['num_papers']}")
print(f"     Pages      : {summary['total_pages']}")
print(f"     ColPali    : {summary['colpali_files']} files ({summary['colpali_size_mb']:.1f} MB)")
print(f"     ChromaDB   : {summary['chroma_entries']} entries")

# ─── 3. Create indices zip ───
print(f"\n  [3/4] Creating sci-rag-indices.zip...")
zip_start  = time.time()
zip_path   = "/content/sci-rag-indices.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Add all indices
    for root, dirs, files in os.walk("data/indices"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname   = os.path.relpath(file_path, "data")
            zf.write(file_path, arcname)

    # Add page list json
    page_list = {
        pk: {
            "doc_id"     : meta["doc_id"],
            "page_num"   : meta["page_num"],
            "image_path" : meta["image_path"],
            "paper_title": meta.get("paper_title", ""),
        }
        for pk, meta in page_metadata.items()
    }
    zf.writestr("parsed/page_list.json", json.dumps(page_list, indent=2))

zip_size = os.path.getsize(zip_path) / 1024**2
print(f"  ✅ sci-rag-indices.zip : {zip_size:.1f} MB")
print(f"     Time: {time.time()-zip_start:.1f}s")

# ─── 4. Create pages zip ───
print(f"\n  [4/4] Creating sci-rag-pages.zip...")
pages_start    = time.time()
pages_zip_path = "/content/sci-rag-pages.zip"

with zipfile.ZipFile(pages_zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("data/parsed"):
        for file in files:
            file_path = os.path.join(root, file)
            arcname   = os.path.relpath(file_path, "data")
            zf.write(file_path, arcname)

pages_zip_size = os.path.getsize(pages_zip_path) / 1024**2
print(f"  ✅ sci-rag-pages.zip   : {pages_zip_size:.1f} MB")
print(f"     Time: {time.time()-pages_start:.1f}s")

# ─── Final Summary ───
print(f"\n{'='*60}")
print(f"  ALL FILES SAVED!")
print(f"{'='*60}")
print(f"""
  Files ready to download:
  ├── /content/sci-rag-indices.zip  ({zip_size:.0f} MB)
  │   ├── indices/multivectors/   (182 ColPali .npy files)
  │   ├── indices/chroma_index/   (SciNCL ChromaDB)
  │   ├── indices/doc_mapping.json
  │   ├── indices/page_metadata.json
  │   └── indices/summary.json
  │
  └── /content/sci-rag-pages.zip   ({pages_zip_size:.0f} MB)
      ├── parsed/pages/            (182 PNG images)
      └── parsed/markdown/         (10 markdown files)
""")
print("✅ CELL 7 COMPLETE!")

  CELL 7: SAVE + ZIP ALL INDICES

  [1/4] Index Check:
  ColPali .npy files : 182
  Page metadata      : 182
  Doc mapping        : 10

  [2/4] Saving summary...
  ✅ Summary saved!
     Papers     : 10
     Pages      : 182
     ColPali    : 182 files (91.6 MB)
     ChromaDB   : 182 entries

  [3/4] Creating sci-rag-indices.zip...
  ✅ sci-rag-indices.zip : 55.4 MB
     Time: 24.6s

  [4/4] Creating sci-rag-pages.zip...
  ✅ sci-rag-pages.zip   : 109.8 MB
     Time: 6.6s

  ALL FILES SAVED!

  Files ready to download:
  ├── /content/sci-rag-indices.zip  (55 MB)
  │   ├── indices/multivectors/   (182 ColPali .npy files)
  │   ├── indices/chroma_index/   (SciNCL ChromaDB)
  │   ├── indices/doc_mapping.json
  │   ├── indices/page_metadata.json
  │   └── indices/summary.json
  │
  └── /content/sci-rag-pages.zip   (110 MB)
      ├── parsed/pages/            (182 PNG images)
      └── parsed/markdown/         (10 markdown files)

✅ CELL 7 COMPLETE!


In [4]:

# ═══════════════════════════════════════════════════════════════
# CELL 8: Quick Verification Test
# Tests ColPali + SciNCL + Score Fusion together
# ═══════════════════════════════════════════════════════════════

import os
import gc
import json
import time
import numpy as np
import torch
from PIL import Image
from colpali_engine.models import ColPali, ColPaliProcessor
from sentence_transformers import SentenceTransformer
import chromadb

# ─── Config ───
NPY_DIR    = "data/indices/multivectors"
CHROMA_DIR = "data/indices/chroma_index"

# ─── Load metadata ───
with open("data/indices/page_metadata.json", "r") as f:
    page_metadata = json.load(f)

with open("data/indices/summary.json", "r") as f:
    summary = json.load(f)

print("=" * 60)
print("  CELL 8: VERIFICATION TEST")
print("=" * 60)
print(f"  Papers    : {summary['num_papers']}")
print(f"  Pages     : {summary['total_pages']}")
print(f"  ColPali   : {summary['colpali_files']} .npy files")
print(f"  ChromaDB  : {summary['chroma_entries']} entries")
print("=" * 60)

# Test query
query = "How do Vision Transformers handle image patches?"

# ═══════════════════════════════════════
# TEST 1: ColPali Visual Retrieval
# ═══════════════════════════════════════
print(f"\n  [1/3] ColPali Visual Retrieval Test...")

colpali_model = ColPali.from_pretrained(
    "vidore/colpali-v1.2",
    torch_dtype=torch.float16,
    device_map="cuda",
    low_cpu_mem_usage=True,
)
colpali_processor = ColPaliProcessor.from_pretrained("vidore/colpali-v1.2")
colpali_model.eval()

# Encode query
query_batch = colpali_processor.process_queries(queries=[query])
query_batch = {k: v.to(colpali_model.device) for k, v in query_batch.items()}

with torch.no_grad():
    query_emb = colpali_model(**query_batch)

query_vec = query_emb[0].cpu().float().numpy()  # shape: (num_tokens, dim)

del query_batch, query_emb
torch.cuda.empty_cache()

# Score all pages
print(f"  Scoring {len(page_metadata)} pages against query...")
scores = []

for page_key in list(page_metadata.keys()):
    npy_path = os.path.join(NPY_DIR, f"{page_key}.npy")
    if not os.path.exists(npy_path):
        continue
    page_vec = np.load(npy_path)  # shape: (num_patches, dim)

    # MaxSim scoring (ColPali style)
    # query_vec: (qt, dim), page_vec: (pt, dim)
    sim_matrix = np.matmul(query_vec, page_vec.T)  # (qt, pt)
    score      = float(sim_matrix.max(axis=1).mean())
    scores.append((page_key, score))

scores.sort(key=lambda x: x[1], reverse=True)

print(f"\n  Query: '{query}'")
print(f"  Top 5 ColPali results:")
for pk, sc in scores[:5]:
    meta  = page_metadata.get(pk, {})
    title = meta.get("paper_title", pk)
    print(f"    {pk}: score={sc:.4f} | {title[:45]}")

# Unload ColPali
del colpali_model, colpali_processor
gc.collect()
torch.cuda.empty_cache()

# ═══════════════════════════════════════
# TEST 2: SciNCL Text Retrieval
# ═══════════════════════════════════════
print(f"\n  [2/3] SciNCL Text Retrieval Test...")

scincl = SentenceTransformer("malteos/scincl", device="cuda")
query_emb_scincl = scincl.encode(query, convert_to_numpy=True)

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection    = chroma_client.get_collection("sci_rag_pages")

results = collection.query(
    query_embeddings=[query_emb_scincl.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

print(f"  Query: '{query}'")
print(f"  Top 5 SciNCL results:")
for i, doc_id in enumerate(results["ids"][0]):
    meta  = results["metadatas"][0][i]
    dist  = results["distances"][0][i]
    sim   = 1 - dist
    title = meta.get("paper_title", doc_id)
    print(f"    {doc_id}: sim={sim:.3f} | {title[:45]}")

del scincl
gc.collect()
torch.cuda.empty_cache()

# ═══════════════════════════════════════
# TEST 3: Score Fusion (0.7 + 0.3)
# ═══════════════════════════════════════
print(f"\n  [3/3] Score Fusion (0.7 ColPali + 0.3 SciNCL)...")

# Normalize ColPali scores
c_scores = {pk: sc for pk, sc in scores[:20]}  # top 20
c_vals   = list(c_scores.values())
c_min, c_max = min(c_vals), max(c_vals)

# Normalize SciNCL scores
s_scores = {}
for i, doc_id in enumerate(results["ids"][0]):
    s_scores[doc_id] = 1 - results["distances"][0][i]
s_vals   = list(s_scores.values())
s_min, s_max = min(s_vals), max(s_vals)

# Fuse
fused = {}
for pk, sc in c_scores.items():
    norm_c     = (sc - c_min) / (c_max - c_min + 1e-8)
    fused[pk]  = 0.7 * norm_c

for pk, sc in s_scores.items():
    norm_s = (sc - s_min) / (s_max - s_min + 1e-8)
    if pk in fused:
        fused[pk] += 0.3 * norm_s
    else:
        fused[pk]  = 0.3 * norm_s

fused_sorted = sorted(fused.items(), key=lambda x: x[1], reverse=True)

print(f"  Top 5 Fused results:")
for pk, sc in fused_sorted[:5]:
    meta  = page_metadata.get(pk, {})
    title = meta.get("paper_title", pk)
    print(f"    {pk}: fused={sc:.4f} | {title[:45]}")

# ─── Final Summary ───
print(f"\n{'='*60}")
print(f"  ✅ VERIFICATION PASSED!")
print(f"  ColPali : {len(scores)} pages scored")
print(f"  SciNCL  : {collection.count()} pages in ChromaDB")
print(f"  Fusion  : Both working together!")
print(f"{'='*60}")
print("✅ CELL 8 COMPLETE!")

  CELL 8: VERIFICATION TEST
  Papers    : 10
  Pages     : 182
  ColPali   : 182 .npy files
  ChromaDB  : 182 entries

  [1/3] ColPali Visual Retrieval Test...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/605 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] ColPali LOAD REPORT from: vidore/colpali-v1.2
Key                                                                               | Status     | 
----------------------------------------------------------------------------------+------------+-
model.language_model.model.layers.{0...17}.mlp.gate_proj.lora_B.default.weight    | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.o_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.down_proj.lora_B.default.weight    | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.q_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.k_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.self_attn.v_proj.lora_B.default.weight | UNEXPECTED | 
model.language_model.model.layers.{0...17}.mlp.up_proj.lo

  Scoring 182 pages against query...

  Query: 'How do Vision Transformers handle image patches?'
  Top 5 ColPali results:
    2108.00102_page_2: score=0.7026 | DeepViT: Towards Deeper Vision Transformer
    2106.10270_page_1: score=0.6968 | Training data-efficient image transformers & 
    2203.14465_page_15: score=0.6837 | EfficientFormer: Vision Transformers at Mobil
    2106.10270_page_5: score=0.6815 | Training data-efficient image transformers & 
    2010.11929_page_14: score=0.6801 | An Image is Worth 16x16 Words: Transformers f

  [2/3] SciNCL Text Retrieval Test...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Query: 'How do Vision Transformers handle image patches?'
  Top 5 SciNCL results:
    2010.11929_page_1: sim=0.891 | An Image is Worth 16x16 Words: Transformers f
    2111.09883_page_2: sim=0.890 | Aggregating Nested Transformers
    2205.01580_page_1: sim=0.887 | A Battle of Architectures: ViT, ResNet and Hy
    2010.11929_page_4: sim=0.886 | An Image is Worth 16x16 Words: Transformers f
    2106.10270_page_4: sim=0.878 | Training data-efficient image transformers & 

  [3/3] Score Fusion (0.7 ColPali + 0.3 SciNCL)...
  Top 5 Fused results:
    2108.00102_page_2: fused=0.7000 | DeepViT: Towards Deeper Vision Transformer
    2106.10270_page_1: fused=0.5843 | Training data-efficient image transformers & 
    2203.14465_page_15: fused=0.3237 | EfficientFormer: Vision Transformers at Mobil
    2010.11929_page_1: fused=0.3000 | An Image is Worth 16x16 Words: Transformers f
    2106.10270_page_5: fused=0.2803 | Training data-efficient image transformers & 

  ✅ VERIFICATION PASSED!
  ColP

In [5]:

# ═══════════════════════════════════════════════════════════════
# CELL 9: Download All Files to Your PC
# ═══════════════════════════════════════════════════════════════

from google.colab import files
import os

print("=" * 60)
print("  CELL 9: DOWNLOADING FILES TO YOUR PC")
print("=" * 60)

# Check files exist
indices_zip = "/content/sci-rag-indices.zip"
pages_zip   = "/content/sci-rag-pages.zip"

idx_size  = os.path.getsize(indices_zip) / 1024**2
page_size = os.path.getsize(pages_zip)   / 1024**2

print(f"\n  Files ready:")
print(f"  ├── sci-rag-indices.zip : {idx_size:.1f} MB")
print(f"  └── sci-rag-pages.zip   : {page_size:.1f} MB")
print(f"  Total                  : {idx_size + page_size:.1f} MB")
print(f"\n  Starting downloads...")
print(f"  (Browser will ask where to save each file)\n")

# Download 1
print("  📥 Downloading sci-rag-indices.zip...")
files.download(indices_zip)

# Download 2
print("  📥 Downloading sci-rag-pages.zip...")
files.download(pages_zip)

print(f"\n{'='*60}")
print(f"  ✅ DOWNLOADS COMPLETE!")
print(f"{'='*60}")
print(f"""
  Save these files safely on your PC!

  What's inside:
  ├── sci-rag-indices.zip  ({idx_size:.0f} MB)
  │   ├── multivectors/      ← 182 ColPali .npy files
  │   ├── chroma_index/      ← SciNCL ChromaDB
  │   ├── doc_mapping.json   ← paper metadata
  │   ├── page_metadata.json ← page metadata
  │   └── summary.json       ← index stats
  │
  └── sci-rag-pages.zip    ({page_size:.0f} MB)
      ├── pages/             ← 182 PNG images
      └── markdown/          ← 10 paper texts
""")
print("✅ CELL 9 COMPLETE!")
print("\n🎉 OFFLINE PIPELINE 100% DONE!")
print("   Next step: Online Pipeline (RAG Query System)!")

  CELL 9: DOWNLOADING FILES TO YOUR PC

  Files ready:
  ├── sci-rag-indices.zip : 55.4 MB
  └── sci-rag-pages.zip   : 109.8 MB
  Total                  : 165.2 MB

  Starting downloads...
  (Browser will ask where to save each file)

  📥 Downloading sci-rag-indices.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  📥 Downloading sci-rag-pages.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  ✅ DOWNLOADS COMPLETE!

  Save these files safely on your PC!
  
  What's inside:
  ├── sci-rag-indices.zip  (55 MB)
  │   ├── multivectors/      ← 182 ColPali .npy files
  │   ├── chroma_index/      ← SciNCL ChromaDB
  │   ├── doc_mapping.json   ← paper metadata
  │   ├── page_metadata.json ← page metadata  
  │   └── summary.json       ← index stats
  │
  └── sci-rag-pages.zip    (110 MB)
      ├── pages/             ← 182 PNG images
      └── markdown/          ← 10 paper texts

✅ CELL 9 COMPLETE!

🎉 OFFLINE PIPELINE 100% DONE!
   Next step: Online Pipeline (RAG Query System)!
